# Learning Gymnasium — LunarLander-v3

This notebook walks through the Gymnasium API step by step using LunarLander-v3 as the example environment.

**Resources:**
- [Sutton & Barto — RL textbook (free PDF)](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)
- [Gymnasium docs — CarRacing](https://gymnasium.farama.org/environments/box2d/car_racing/#starting-state)
- [RL-Adventure (reference implementations)](https://github.com/higgsfield/RL-Adventure)
- [CleanRL — PPO implementation](https://docs.cleanrl.dev/rl-algorithms/ppo/)

---

### What is LunarLander?

A classic control problem: land a spacecraft on a pad between two flags without crashing.

| Property | Value |
|---|---|
| **Observation** | 8 floats — position, velocity, angle, angular velocity, leg contact |
| **Action space** | Discrete(4) — do nothing, left engine, main engine, right engine |
| **Reward** | +100–140 for landing, −100 for crash, −0.3/frame for main engine |
| **Solved** | Score ≥ 200 averaged over 100 episodes |

---
## Step 1 — Imports

- `gymnasium` — the environment
- `matplotlib` — rendering frames as images and plotting
- `numpy` — numerical operations on observations
- `FuncAnimation` + `HTML` — building an inline animation from collected frames

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

---
## Step 2 — Create the environment

`gym.make()` loads the environment by name. We pass `render_mode="rgb_array"` so that
`env.render()` returns frames as NumPy arrays we can display in the notebook (rather than
opening a GUI window).

After creating it, `env.reset()` starts a fresh episode and returns:
- `obs` — the first observation
- `info` — auxiliary data (usually empty at reset)

In [ ]:
env = gym.make("LunarLander-v3", render_mode="rgb_array")
obs, info = env.reset(seed=42)

print("Environment ready.")
print()
print("Observation space:", env.observation_space)
# Box of 8 floats — position (x,y), velocity (vx,vy), angle, angular vel, leg contacts

print("Action space:     ", env.action_space)
# Discrete(4): 0=do nothing  1=left engine  2=main engine  3=right engine

---
## Step 3 — Inspect the observation

Unlike CarRacing (which gives a pixel image), LunarLander gives a **vector** of 8 floats.
These are the physics state of the lander — no image processing needed.

| Index | Meaning | Range |
|---|---|---|
| 0 | x position | −2.5 → 2.5 |
| 1 | y position | −2.5 → 2.5 |
| 2 | x velocity | −10 → 10 |
| 3 | y velocity | −10 → 10 |
| 4 | angle | −π → π |
| 5 | angular velocity | −10 → 10 |
| 6 | left leg contact | 0 or 1 |
| 7 | right leg contact | 0 or 1 |

In [ ]:
print("obs.shape :", obs.shape)   # (8,)
print("obs.dtype :", obs.dtype)   # float32
print()
labels = ["x pos", "y pos", "x vel", "y vel", "angle", "ang vel", "leg L", "leg R"]
for label, val in zip(labels, obs):
    print(f"  {label:<10} {val: .4f}")

---
## Step 4 — Take one step

The core RL loop is:
```
action  →  env.step(action)  →  (next_obs, reward, terminated, truncated, info)
```

| Return value | Meaning |
|---|---|
| `obs` | Next observation after the action |
| `reward` | Score change this step |
| `terminated` | Episode ended naturally (landed or crashed) |
| `truncated` | Episode cut off by a step limit |
| `info` | Extra diagnostic data |

In [ ]:
action = env.action_space.sample()   # pick a random action
obs, reward, terminated, truncated, info = env.step(action)

action_names = ["do nothing", "left engine", "main engine", "right engine"]
print(f"Action taken:  {action}  ({action_names[action]})")
print(f"Reward:        {reward:.4f}")
print(f"Terminated:    {terminated}  |  Truncated: {truncated}")
print(f"Info:          {info}")

---
## Step 5 — Render a single frame

`env.render()` returns the current frame as a `(H, W, 3)` uint8 NumPy array.
We pass it straight to `plt.imshow()` to display it in the notebook.

In [ ]:
frame = env.render()

print("Frame shape:", frame.shape)   # e.g. (400, 600, 3)

plt.figure(figsize=(6, 4))
plt.imshow(frame)
plt.axis("off")
plt.title("LunarLander — current frame")
plt.show()

---
## Step 6 — Animate a full episode (random policy)

We run the environment until the episode ends (or a frame cap), collecting each rendered frame.
Then we stitch them into an inline animation using `FuncAnimation`.

A random policy will crash almost every time — that's expected. The goal here is to
verify the full loop works before adding a trained agent.

In [ ]:
env = gym.make("LunarLander-v3", render_mode="rgb_array")
obs, info = env.reset(seed=42)

frames       = []
rewards      = []
terminated   = False
truncated    = False

# ── Main loop ──────────────────────────────────────────────────
while not (terminated or truncated):
    frames.append(env.render())               # capture frame before the step
    action = env.action_space.sample()        # random policy
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
# ───────────────────────────────────────────────────────────────

env.close()
print(f"Episode finished after {len(frames)} frames")
print(f"Total reward : {sum(rewards):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.axis("off")
img = ax.imshow(frames[0])

def update(frame):
    img.set_data(frame)
    return [img]

ani = FuncAnimation(fig, update, frames=frames, interval=30, blit=True)
plt.close(fig)   # prevent a static extra image appearing
HTML(ani.to_jshtml())

---
## Step 7 — Reward curve

Plot per-step reward to understand what the environment is signalling.
With a random policy you'll see mostly negative rewards (crashing, firing engines randomly)
and a large negative spike at the end when the lander crashes.

In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(rewards, color="steelblue", linewidth=1)
plt.axhline(0, color="red", linewidth=0.8, linestyle="--", label="zero")
plt.xlabel("Step")
plt.ylabel("Reward")
plt.title(f"Per-step reward — random policy  (total={sum(rewards):.1f})")
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 8 — The canonical Gymnasium loop

This is the minimal pattern from the Gymnasium docs — run multiple episodes,
resetting when each one ends. This is the skeleton every RL training loop is built on.

Swap `env.action_space.sample()` with your policy's `predict(obs)` call to start training.

In [ ]:
env = gym.make("LunarLander-v3", render_mode="rgb_array")
observation, info = env.reset(seed=42)

for step in range(1000):
    action = env.action_space.sample()   # ← replace with your policy here

    observation, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        observation, info = env.reset()  # start a new episode

env.close()
print("Done — ran 1000 steps across multiple episodes.")

---
## Summary

| Concept | Detail |
|---|---|
| `gym.make(id, render_mode="rgb_array")` | Load environment in headless mode |
| `env.reset(seed=42)` | Start a new episode, returns first obs |
| Observation (LunarLander) | 8 floats — physics state vector |
| `env.action_space` | `Discrete(4)` — 4 possible actions |
| `env.step(action)` | Advance one timestep, returns obs/reward/done flags |
| `env.render()` | Returns current frame as NumPy array |
| `terminated` | Episode ended naturally (landed or crashed) |
| `truncated` | Episode cut short by step limit |

**Key difference vs CarRacing:** LunarLander's observation is a state **vector** (8 floats),
not a pixel image — so you can use a simple MLP instead of a CNN.